### Experiments: BPE tokenizer for PAWS-Wiki

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


## 1. Training the tokenizer

Our classifier needs a way to segment text into tokens. We train a **Byte-Pair Encoding (BPE)** tokenizer on the PAWS training set.

BPE starts from single characters and repeatedly merges the most frequent pair of symbols until it reaches the target vocabulary size. Frequent words end up as a single token, while rare words (e.g. names) are split into sub-word pieces, so we almost never need an `[UNK]` token.

**Training data.** We use both sentences of every pair in the **train** split only (98,802 sentences), so no information from validation or test leaks into the tokenizer. PAWS is built from Wikipedia, so it is large enough for a vocabulary of this size.

In [14]:
from src.dataset import load_paws, tokenizer_corpus
from src.tokenization import train_tokenizer

dataset = load_paws()
corpus = tokenizer_corpus(dataset)   # train only, sentence1 + sentence2
print(f"{len(corpus):,} sentences in the tokenizer corpus")

Removed 63 examples from 31 conflicting pairs.
Training examples remaining: 49,338
98,676 sentences in the tokenizer corpus


## 2. Choosing the hyperparameters

Two hyperparameters:
- `vocab_size`: number of tokens in the vocabulary.
- `min_frequency`: a pair of symbols is only merged if it appears at least this many times.

We train several tokenizers and measure, on the **validation** sentences (unseen during training):
- `avg_tokens_per_sentence`: fewer tokens means the vocabulary groups more characters per token.
- `unk_rate`: fraction of tokens that are `[UNK]`. Close to 0 means the tokenizer covers new text.

Larger vocabularies give shorter sequences but sparser features, since many tokens appear only a few times.

In [15]:
import pandas as pd
from src.tokenization import tokenizer_stats

val_texts = list(dataset["validation"]["sentence1"]) + list(dataset["validation"]["sentence2"])

# Train each tokenizer once and keep it, so the following cells can reuse it
tokenizers = {}
rows = []
for vocab_size in [1000, 4000, 8000]:
    for min_freq in [2, 5]:
        tokenizers[(vocab_size, min_freq)] = train_tokenizer(corpus, vocab_size, min_freq)
        rows.append({"min_frequency": min_freq, **tokenizer_stats(tokenizers[(vocab_size, min_freq)], val_texts)})
pd.DataFrame(rows)

,min_frequency,vocab_size,avg_tokens_per_sentence,unk_rate
0,2,1000,44.898125,0.000063
1,5,1000,44.898125,0.000063
2,2,4000,33.070750,0.000085
3,5,4000,33.070750,0.000085
4,2,8000,29.212563,0.000096
5,5,8000,29.212563,0.000096


**Observations.**
- `min_frequency` has **no effect** here (2 and 5 give identical rows): with these vocabulary sizes every merge already appears far more than 5 times, so the filter never applies and training stops when `vocab_size` is reached. We keep `min_frequency=2`, as in the tutorial.
- `vocab_size` matters: going from 1,000 (the tutorial's value) to 4,000 shortens sentences considerably (≈45 → ≈33 tokens), and 8,000 gives a smaller further gain (≈29).
- `[UNK]` is practically zero in all cases (<0.02%), so coverage is not a differentiating factor.

These statistics only describe the tokenizers; they do not tell us which one classifies best. That is decided later with the classifier's validation accuracy.

### 2.1. Inspecting how sentences are tokenized

The table above averages over the whole validation set. To see *how* text is actually split, we look at three validation sentences with each vocabulary size. `▁` marks the start of a word. We look for: are frequent words kept whole? Are rare words (names) split into meaningful pieces or into single characters?

In [16]:
for i in [0, 5, 10]:
    sentence = dataset["validation"][i]["sentence1"]
    print(sentence)
    for vocab_size in [1000, 4000, 8000]:
        tokens = tokenizers[(vocab_size, 2)].encode(sentence).tokens
        print(f"  vocab={vocab_size:>5} ({len(tokens):>2} tokens): {' '.join(tokens)}")
    print()

Bradd Crellin represented BARLA Cumbria on a tour of Australia with 6 other players representing Britain , also on a tour of Australia .
  vocab= 1000 (48 tokens): ▁B rad d ▁C re l lin ▁rep resent ed ▁B A R L A ▁C um b ri a ▁on ▁a ▁t our ▁of ▁Austral ia ▁with ▁6 ▁other ▁play ers ▁rep resent ing ▁B rit ain ▁, ▁also ▁on ▁a ▁t our ▁of ▁Austral ia ▁.
  vocab= 4000 (35 tokens): ▁Brad d ▁C re l lin ▁represented ▁B A R L A ▁C umb ria ▁on ▁a ▁tour ▁of ▁Australia ▁with ▁6 ▁other ▁players ▁represent ing ▁Britain ▁, ▁also ▁on ▁a ▁tour ▁of ▁Australia ▁.
  vocab= 8000 (33 tokens): ▁Brad d ▁Cre l lin ▁represented ▁B A R LA ▁C umb ria ▁on ▁a ▁tour ▁of ▁Australia ▁with ▁6 ▁other ▁players ▁represent ing ▁Britain ▁, ▁also ▁on ▁a ▁tour ▁of ▁Australia ▁.

Kathy and her husband Pete Beale ( Peter Dean ) are stable financially .
  vocab= 1000 (30 tokens): ▁K ath y ▁and ▁her ▁h us b and ▁P et e ▁Be ale ▁( ▁P eter ▁De an ▁) ▁are ▁st able ▁f in an c ial ly ▁.
  vocab= 4000 (21 tokens): ▁Kath y ▁and ▁her ▁husba

**What to look for.** With a small vocabulary even common words are split (e.g. *husband*), which loses information; with a large one almost every word is a token, but many tokens are rare, so each has few training examples. Rare proper names are split in all cases.

## 3. Final tokenizer

In [17]:
# Final tokenizer: reuse the one already trained above (no need to train it again)
final_vocab_size = 8000   # change if the classifier comparison favours another size
tokenizer = tokenizers[(final_vocab_size, 2)]
tokenizer.save(f"tokenizer_bpe_{final_vocab_size}.json")
print("vocabulary size:", tokenizer.get_vocab_size())

vocabulary size: 8000


## 4. Representation spaces

An encoder maps one input to a vector. A PAWS input is a *pair* of sentences, so we encode each sentence with a base encoder into vectors `a` and `b`, and `PairEncoder` combines them as `[|a - b|, a * b]`:
- `|a - b|` is non-zero for tokens that differ between the two sentences,
- `a * b` is non-zero for tokens that both sentences share.

This lets a linear model compare the sentences directly, which simply concatenating `[a, b]` does not, and it is symmetric (swapping the sentences gives the same vector).

| Name | Base encoder (per sentence) |
|---|---|
| `TokenIndicator` | 1 if the token appears in the sentence, else 0 |
| `BagOfWords` | number of times each token appears in the sentence |

In [18]:
from collections import OrderedDict
from src.encoders import TokenIndicator, BagOfWords, PairEncoder, NgramBagOfWords

encoders = {
    "TokenIndicator": PairEncoder(TokenIndicator(tokenizer)),
    "BagOfWords":     PairEncoder(BagOfWords(tokenizer)),
    "NGram": PairEncoder(NgramBagOfWords(tokenizer, n = 4, include_lower=False).fit(corpus)),

}

encoders_concat = {
    "TokenIndicator": PairEncoder(TokenIndicator(tokenizer), mode="concat"),
    "BagOfWords":     PairEncoder(BagOfWords(tokenizer), mode="concat"),
}

## 5. Cross-validation setup

Each PAWS example is a **pair** of sentences, so an input `x` is the tuple `(sentence1, sentence2)`. The code from the tutorial works with `(input, label)` pairs, so we build them in that format.

We run 5-fold cross-validation on the **train** split: the data is shuffled and split into 5 parts, and each model is trained 5 times, each time evaluating on a different part. The validation and test splits of PAWS are left untouched. The seed makes the shuffling reproducible, and all models use exactly the same folds so that the comparison is fair.

In [19]:
from src import experiment, clslib, analysis
from src.dataset import paws_pairs

outcome_space = clslib.NamedClasses(["non-paraphrase", "paraphrase"])
results_folder = "results"

my_seeds = [0]
num_folds = 5
my_cross_validation_sets = [
    experiment.cross_validation_split(paws_pairs(dataset["train"]), k=num_folds, seed=seed)
    for seed in my_seeds
]

## 6. Logistic regression (log-linear model)

We train one logistic regression per representation space and per fold, and record the accuracy on the held-out fold.

### 6.1. Tuning the logistic regression

The regularization strength is controlled by `C` (smaller `C` = stronger regularization). With thousands of features and weak signal, a stronger penalty may reduce overfitting. We train the same model with several values of `C`, for every representation and on the same folds, and keep each one as a separate system so they can be compared later (`logistic_<encoder>` above is `C=1`).

In [20]:
C_values = [0.001, 0.01, 0.1, 1]          # C=1 is the run above

my_classifiers = OrderedDict()
for enc_name, enc in encoders.items():
    for C in C_values:
        name = f"logistic_{enc_name}_C{C}"
        my_classifiers[name] = [
            experiment.cross_validation_experiment(
                data_splits=my_cross_validation_sets[run],
                outcome_space=outcome_space,
                encoder=enc,
                model_option="logistic",
                save_to=f"{results_folder}/{name}/seed-{seed}",
                loglin_args={"max_iter": 100, "C": C},
            )
            for run, seed in enumerate(my_seeds)
        ]

# mean accuracy over folds for every logistic system
for name, runs in my_classifiers.items():
    if name.startswith("logistic"):
        accs = [a for r in runs for a in r["accuracies"].values()]
        print(f"{name:35s} {sum(accs)/len(accs):.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ 

  0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ 

  0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and wi

... (15 outputs left out)

c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and wi

  0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ 

logistic_TokenIndicator_C0.001      0.5669
logistic_TokenIndicator_C0.01       0.5735
logistic_TokenIndicator_C0.1        0.5144
logistic_TokenIndicator_C1          0.4549
logistic_BagOfWords_C0.001          0.5756
logistic_BagOfWords_C0.01           0.5642
logistic_BagOfWords_C0.1            0.5521
logistic_BagOfWords_C1              0.5501
logistic_NGram_C0.001               0.5591
logistic_NGram_C0.01                0.5765
logistic_NGram_C0.1                 0.5838
logistic_NGram_C1                   0.5657


## 7. MLP (one hidden layer)

Same experiment, replacing the logistic regression with a neural network with one hidden layer (128 units, ReLU, dropout 0.1).

In [ ]:
mlp_args = {"hidden_dim": 128, "dropout": 0.1, "batch_size": 64,
            "learning_rate": 1e-4, "weight_decay": 1e-4, "num_epochs": 3}

for enc_name, enc in encoders.items():
    name = f"mlp_{enc_name}"
    my_classifiers[name] = [
        experiment.cross_validation_experiment(
            data_splits=my_cross_validation_sets[run],
            outcome_space=outcome_space,
            encoder=enc,
            model_option="mlp",
            save_to=f"{results_folder}/{name}/seed-{seed}",
            smlp_args=mlp_args,
        )
        for run, seed in enumerate(my_seeds)
    ]

analysis.plot_box_and_points(analysis.make_data_frame(my_classifiers))

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 16.734 seconds
Dataloader done! Time taken: 0.014 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\src\smlp.py:69: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:252.)
  indices = torch.tensor([x.row, x.col], dtype=torch.long)
c:\Users\nuria\Desktop\MASTER\NLP\Practicals\Practica 1\nlp1\src\smlp.py:72: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\Context.cpp:923.)
  return torch.sparse_coo_tensor(indices, values, size=x.shape).

Epoch 01 | train loss: 0.6859 | train acc: 0.5638 | val loss: 0.6863 | val acc: 0.5762
Epoch 02 | train loss: 0.6713 | train acc: 0.6007 | val loss: 0.6910 | val acc: 0.5676
Epoch 03 | train loss: 0.6526 | train acc: 0.6195 | val loss: 0.7034 | val acc: 0.5455
Epoch 04 | train loss: 0.6361 | train acc: 0.6303 | val loss: 0.7210 | val acc: 0.5346
Epoch 05 | train loss: 0.6231 | train acc: 0.6377 | val loss: 0.7420 | val acc: 0.5130
Epoch 06 | train loss: 0.6125 | train acc: 0.6426 | val loss: 0.7644 | val acc: 0.4613
Epoch 07 | train loss: 0.6035 | train acc: 0.6461 | val loss: 0.7822 | val acc: 0.4590
Epoch 08 | train loss: 0.5962 | train acc: 0.6513 | val loss: 0.8048 | val acc: 0.4729
Epoch 09 | train loss: 0.5894 | train acc: 0.6535 | val loss: 0.8272 | val acc: 0.4443
Epoch 10 | train loss: 0.5836 | train acc: 0.6557 | val loss: 0.8481 | val acc: 0.4334
Training done! Time taken: 426.127 seconds
Transforming documents...
Document transformation done! Time taken: 20.080 seconds
Data

  0%|          | 0/10 [00:00<?, ?it/s]

... (8 outputs left out)

Transforming documents...
Document transformation done! Time taken: 56.965 seconds
Dataloader done! Time taken: 0.006 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6851 | train acc: 0.5706 | val loss: 0.6853 | val acc: 0.5742
Epoch 02 | train loss: 0.6706 | train acc: 0.6005 | val loss: 0.6913 | val acc: 0.5686


KeyboardInterrupt: 

### 7.1. MLP with tuned hyperparameters

The training log of the basic MLP shows overfitting: training accuracy keeps rising while validation loss gets worse after the first epoch, and the best epoch is the first one. We keep the same architecture (one hidden layer) and change only the hyperparameters that reduce overfitting:
- `dropout` 0.1 -> 0.5 and `weight_decay` 1e-4 -> 1e-3: stronger regularization.
- `hidden_dim` 128 -> 64: fewer parameters to memorize the training set.
- `num_epochs` 10 -> 3: the best epoch is always among the first ones, so more epochs only cost time.

To limit the cost we use only the two representations that did best with the logistic regression.


In [ ]:
import numpy as np

tune_encoder = "BagOfWords"

variants = {
    "dropout0.5": {**mlp_args, "num_epochs": 3, "dropout": 0.5},
    "wd1e-3":     {**mlp_args, "num_epochs": 3, "weight_decay": 1e-3},
    "hidden64":   {**mlp_args, "num_epochs": 3, "hidden_dim": 64},
    "lr1e-3":     {**mlp_args, "num_epochs": 3, "learning_rate": 1e-3},
}

# reference: the basic MLP already trained in section 7 (same folds)
base_accs = list(my_classifiers[f"mlp_{tune_encoder}"][0]["accuracies"].values())

tuning_results = {}
for tag, args in variants.items():
    name = f"tune_{tune_encoder}_{tag}"
    tuning_results[tag] = experiment.cross_validation_experiment(
        data_splits=my_cross_validation_sets[0],
        outcome_space=outcome_space,
        encoder=encoders[tune_encoder],
        model_option="mlp",
        save_to=f"{results_folder}/{name}/seed-{my_seeds[0]}",
        smlp_args=args,
    )

rows = [{"config": "basic MLP", "mean acc": np.mean(base_accs), "std": np.std(base_accs)}]
for tag, res in tuning_results.items():
    accs = list(res["accuracies"].values())
    rows.append({"config": tag, "mean acc": np.mean(accs), "std": np.std(accs),
                 "diff vs basic": np.mean(accs) - np.mean(base_accs)})
pd.DataFrame(rows).round(4)



## 8. MLP with more hidden layers (extra architecture)

The assignment asks for one architecture of our choice besides the log-linear model and the one-hidden-layer MLP. We extend the MLP with **additional hidden layers**. The first layer still takes the sparse input; every extra layer is dense (`hidden_dim -> hidden_dim`, ReLU, dropout). With `num_hidden_layers=1` the model is exactly the tutorial's MLP.

**Motivation.** A deeper network can combine the token-level features into higher-level ones, which in principle could help to capture interactions between tokens of the two sentences that a single hidden layer misses.

**What we vary.** The regularization (`dropout`, `weight_decay`) is fixed to the values chosen for the tuned one-hidden-layer MLP (section 7.2), so that the only difference between both models is the structure. We try the number of layers (2 or 3) and the hidden width (64 or 128).

We follow the same protocol as before: the same 5 folds, one representation (`BagOfWords`) to choose the structure, 3 epochs, and selection by mean validation accuracy only. The test set is not used.


In [ ]:
import numpy as np

tune_encoder = "BagOfWords"

structure_variants = {
    "L2_h64":  {**mlp_tuned_args, "num_hidden_layers": 2, "hidden_dim": 64},
    "L3_h64":  {**mlp_tuned_args, "num_hidden_layers": 3, "hidden_dim": 64},
    "L2_h128": {**mlp_tuned_args, "num_hidden_layers": 2, "hidden_dim": 128},
}

# reference: the tuned one-hidden-layer MLP (same folds)
ref_accs = list(my_classifiers[f"mlp_tuned_{tune_encoder}"][0]["accuracies"].values())

structure_results = {}
for tag, args in structure_variants.items():
    name = f"structure_{tune_encoder}_{tag}"
    structure_results[tag] = experiment.cross_validation_experiment(
        data_splits=my_cross_validation_sets[0],
        outcome_space=outcome_space,
        encoder=encoders[tune_encoder],
        model_option="mlp",
        save_to=f"{results_folder}/{name}/seed-{my_seeds[0]}",
        smlp_args=args,
    )

rows = [{"config": "tuned MLP (1 layer)", "mean acc": np.mean(ref_accs), "std": np.std(ref_accs)}]
for tag, res in structure_results.items():
    accs = list(res["accuracies"].values())
    rows.append({"config": tag, "mean acc": np.mean(accs), "std": np.std(accs),
                 "diff vs tuned MLP": np.mean(accs) - np.mean(ref_accs)})
pd.DataFrame(rows).round(4)
